-> Get Necessary Libraries <br>
-> Set Environment Variables <br>
-> Prepare Training Data into imagesTr, etc <br>
-> Create imagesTs Folder for inference <br>
-> Figure out Epoch Value Changes <br>

-> Setup training <br>
-> !cp Env Folders into zip <br>
-> Run inference on imagesTs, zip outputs <br>
-> Save Results, Maybe <br>

In [ ]:
!pip -q install nnunet

In [2]:
!git clone https://github.com/MIC-DKFZ/MedNeXt.git mednext

Cloning into 'mednext'...
remote: Enumerating objects: 758, done.
remote: Counting objects: 100% (81/81), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 758 (delta 48), reused 51 (delta 29), pack-reused 677
Receiving objects: 100% (758/758), 565.55 KiB | 14.50 MiB/s, done.
Resolving deltas: 100% (421/421), done.


In [3]:
%cd mednext
!pip -q install -e .

/kaggle/working/mednext


In [ ]:
import re
import os
import json
import torch
import shutil
import random
torch.__version__

/kaggle/working


'2.1.2'

In [ ]:
# Download Datasets

In [ ]:
# create these folders

%env nnUNet_raw_data_base = nnUNet_raw_data_base
%env RESULTS_FOLDER = nnUNet_results
%env nnUNet_preprocessed = nnUNet_preprocessed

env: nnUNet_raw_data_base=nnUNet_raw_data_base
env: RESULTS_FOLDER=nnUNet_results
env: nnUNet_preprocessed=nnUNet_preprocessed


In [ ]:
import os
import shutil

# ↳Dataset2023
#   ↳imagesTr
#     BraTS_0001_0000.nii.gz
#     BraTS_0001_0001.nii.gz
#     BraTS_0001_0002.nii.gz
#     BraTS_0001_0003.nii.gz
#   ↳labelsTr
#     BraTS_0001.nii.gz

# Moving Brain MRI Modalities to nnUNet_raw/Dataset2023/imagesTr
# Moving Segmentation Masks to nnUNet_raw/Dataset2023/labelsTr

case_id = 0

base_path = 'ASNR-MICCAI-BraTS2023-SSA-Challenge-TrainingData_V2/'

for folder in os.listdir(base_path):
    case_id += 1
    for file in os.listdir(base_path + folder):
        if 'seg' in file:
            shutil.copy(f'{base_path}{folder}/{file}', 'nnUNet_raw_data_base/nnUNet_raw_data/Task2023_BraTS/labelsTr')
            os.rename(f'nnUNet_raw_data_base/nnUNet_raw_data/Task2023_BraTS/labelsTr/{file}', f"nnUNet_raw_data_base/nnUNet_raw_data/Task2023_BraTS/labelsTr/BraTS_{case_id:04d}.nii.gz")
        if 't1c' in file or 't1ce' in file:
            shutil.copy(f'{base_path}{folder}/{file}', 'nnUNet_raw_data_base/nnUNet_raw_data/Task2023_BraTS/imagesTr')
            os.rename(f'nnUNet_raw_data_base/nnUNet_raw_data/Task2023_BraTS/imagesTr/{file}', f"nnUNet_raw_data_base/nnUNet_raw_data/Task2023_BraTS/imagesTr/BraTS_{case_id:04d}_0000.nii.gz")
        if 't1n' in file or 't1.nii' in file:
            shutil.copy(f'{base_path}{folder}/{file}', 'nnUNet_raw_data_base/nnUNet_raw_data/Task2023_BraTS/imagesTr')
            os.rename(f'nnUNet_raw_data_base/nnUNet_raw_data/Task2023_BraTS/imagesTr/{file}', f"nnUNet_raw_data_base/nnUNet_raw_data/Task2023_BraTS/imagesTr/BraTS_{case_id:04d}_0001.nii.gz")
        if 't2f' in file or 'flair' in file:
            shutil.copy(f'{base_path}{folder}/{file}', 'nnUNet_raw_data_base/nnUNet_raw_data/Task2023_BraTS/imagesTr')
            os.rename(f'nnUNet_raw_data_base/nnUNet_raw_data/Task2023_BraTS/imagesTr/{file}', f"nnUNet_raw_data_base/nnUNet_raw_data/Task2023_BraTS/imagesTr/BraTS_{case_id:04d}_0002.nii.gz")
        if 't2w' in file or 't2.nii' in file:
            shutil.copy(f'{base_path}{folder}/{file}', 'nnUNet_raw_data_base/nnUNet_raw_data/Task2023_BraTS/imagesTr')
            os.rename(f'nnUNet_raw_data_base/nnUNet_raw_data/Task2023_BraTS/imagesTr/{file}', f"nnUNet_raw_data_base/nnUNet_raw_data/Task2023_BraTS/imagesTr/BraTS_{case_id:04d}_0003.nii.gz")


In [ ]:
# Renaming imagesTr Files to nnU-Net Format

def rename_file(filename, patient_counters):
    pattern = re.compile(r'BraTS-SSA-(\d+)-(\d+)-(\d+)\.nii\.gz')
    match = pattern.match(filename)
    if match:
        patient_id = match.group(1)
        modality_part = match.group(3)

        if patient_id not in patient_counters:
            patient_counters[patient_id] = len(patient_counters)
            
        series_part = str(patient_counters[patient_id]).zfill(4)

        new_filename = f'BraTS{patient_id}_{series_part}_{modality_part}.nii.gz'
        return new_filename
    return None

patient_counters = {}
file_list = sorted(os.listdir('nnUNet_raw_data_base/nnUNet_raw_data/Task2023_BraTS/imagesTr/'))
for filename in file_list:
    new_filename = rename_file(filename, patient_counters)
    if new_filename:
        os.rename(f'nnUNet_raw_data_base/nnUNet_raw_data/Task2023_BraTS/imagesTr/{filename}', f'nnUNet_raw_data_base/nnUNet_raw_data/Task2023_BraTS/imagesTr/{new_filename}')

In [ ]:
import os
import json
import numpy as np
from typing import Tuple

# Paths and setup variables
output_file = 'nnUNet_raw_data_base/nnUNet_raw_data/Task2023_BraTS/dataset.json'
imagesTr = 'nnUNet_raw_data_base/nnUNet_raw_data/Task2023_BraTS/imagesTr'
imagesTs = 'nnUNet_raw_data_base/nnUNet_raw_data/Task2023_BraTS/imagesTs'
labelsTr = 'nnUNet_raw_data_base/nnUNet_raw_data/Task2023_BraTS/labelsTr'
modalities = ("t1c", "t1n", "t2f", "t2w")
labels = {0 : "background", 1 : "TC", 2 : "WT", 3 : "ET"}
dataset_name = 'Task2023_BraTS'

def save_json(obj, file, sort_keys=True):
    with open(file, 'w') as f:
        json.dump(obj, f, indent=4, sort_keys=sort_keys)

def subfiles(folder, suffix, join=True):
    all_files = []
    for root, _, files in os.walk(folder):
        for file in files:
            if file.endswith(suffix):
                if join:
                    all_files.append(os.path.join(root, file))
                else:
                    all_files.append(file)
    return all_files

def get_identifiers_from_splitted_files(folder: str):
    uniques = np.unique([i[:-12] for i in subfiles(folder, suffix='.nii.gz', join=False)])
    return uniques

def generate_dataset_json(output_file: str, imagesTr_dir: str, imagesTs_dir: str, modalities: Tuple,
                          labels: dict, dataset_name: str, sort_keys=True, license: str = "hands off!", dataset_description: str = "",
                          dataset_reference="", dataset_release='0.0'):
    train_identifiers = get_identifiers_from_splitted_files(imagesTr_dir)

    if imagesTs_dir is not None:
        test_identifiers = get_identifiers_from_splitted_files(imagesTs_dir)
    else:
        test_identifiers = []

    json_dict = {}
    json_dict['name'] = dataset_name
    json_dict['description'] = dataset_description
    json_dict['tensorImageSize'] = "4D"
    json_dict['reference'] = dataset_reference
    json_dict['licence'] = license
    json_dict['release'] = dataset_release
    json_dict['modality'] = {str(i): modalities[i] for i in range(len(modalities))}
    json_dict['labels'] = {str(i): labels[i] for i in labels.keys()}

    json_dict['numTraining'] = len(train_identifiers)
    json_dict['numTest'] = len(test_identifiers)
    json_dict['training'] = [
        {'image': f"./imagesTr/{i}.nii.gz", "label": f"./labelsTr/{i}.nii.gz"} for i in train_identifiers]
    json_dict['test'] = [f"./imagesTs/{i}.nii.gz" for i in test_identifiers]

    if not output_file.endswith("dataset.json"):
        print("WARNING: output file name is not dataset.json! This may be intentional or not. You decide. "
              "Proceeding anyways...")
    save_json(json_dict, output_file, sort_keys=sort_keys)

In [22]:
generate_dataset_json(output_file=output_file, 
                      imagesTr_dir=imagesTr, 
                      imagesTs_dir=imagesTs,
                      modalities=modalities,
                      labels=labels, 
                      dataset_name=dataset_name)

In [23]:
!mednextv1_plan_and_preprocess -t 2023 -pl3d ExperimentPlanner3D_v21_customTargetSpacing_1x1x1

BraTS_0001
BraTS_0003
BraTS_0005
BraTS_0007
BraTS_0009
BraTS_0011
BraTS_0013
BraTS_0015
before crop: (4, 155, 240, 240) after crop: (4, 126, 167, 134) spacing: [1. 1. 1.] 

before crop: (4, 155, 240, 240) after crop: (4, 145, 178, 139) spacing: [1. 1. 1.] 

before crop: (4, 155, 240, 240) after crop: (4, 138, 183, 139) spacing: [1. 1. 1.] 

before crop: (4, 155, 240, 240) after crop: (4, 148, 169, 126) spacing: [1. 1. 1.] 

before crop: (4, 155, 240, 240) after crop: (4, 124, 173, 139) spacing: [1. 1. 1.] 

before crop: (4, 155, 240, 240) after crop: (4, 136, 168, 132) spacing: [1. 1. 1.] 
before crop: (4, 155, 240, 240) after crop: (4, 132, 177, 127) spacing: [1. 1. 1.] 


before crop: (4, 155, 240, 240) after crop: (4, 137, 182, 135) spacing: [1. 1. 1.] 

BraTS_0010
BraTS_0012
before crop: (4, 155, 240, 240) after crop: (4, 138, 182, 141) spacing: [1. 1. 1.] 

before crop: (4, 155, 240, 240) after crop: (4, 128, 178, 138) spacing: [1. 1. 1.] 

BraTS_0014
BraTS_0006
BraTS_0008
BraTS_0

In [25]:
!mednextv1_train 3d_fullres nnUNetTrainerV2_MedNeXt_M_kernel3 2023 3 -p nnUNetPlansv2.1_trgSp_1x1x1

###############################################
I am running the following nnUNet: 3d_fullres
My trainer class is:  <class 'nnunet_mednext.training.network_training.MedNeXt.nnUNetTrainerV2_MedNeXt.nnUNetTrainerV2_MedNeXt_M_kernel3'>
For that I will be using the following configuration:
num_classes:  3
modalities:  {0: 't1c', 1: 't1n', 2: 't2f', 3: 't2w'}
use_mask_for_norm OrderedDict([(0, True), (1, True), (2, True), (3, True)])
keep_only_largest_region None
min_region_size_per_class None
min_size_per_class None
normalization_schemes OrderedDict([(0, 'nonCT'), (1, 'nonCT'), (2, 'nonCT'), (3, 'nonCT')])
stages...

stage:  0
{'batch_size': 2, 'num_pool_per_axis': [5, 5, 4], 'patch_size': [128, 128, 128], 'median_patient_size_in_voxels': array([139, 175, 138]), 'current_spacing': array([1., 1., 1.]), 'original_spacing': array([1., 1., 1.]), 'do_dummy_2D_data_aug': False, 'pool_op_kernel_sizes': [[2, 2, 2], [2, 2, 2], [2, 2, 2], [2, 2, 2], [2, 2, 1]], 'conv_kernel_sizes': [[3, 3, 3], [3, 3